### **Part 1 - MapReduce:**

In [ ]:
!pip install mrjob

In [ ]:
%%file hw1_mrA_000000000_000000000.py

from mrjob.job import MRJob
from mrjob.step import MRStep

class CompShow(MRJob):

    def contains_letter(self, title):
        letters = ['j', 'q', 'z']
        title_lower = title.lower()
        for letter in letters:
            if letter in title_lower:
                return True
        return False

    def contains_genre(self, genre):
        genres_list = ['Sitcom', 'Talk', 'Politics', 'Spanish', 'Community', 'Martial arts']
        for g in genres_list:
            if g in genre:
                return True
        return False


    def split_properly(self, line):
              parts = []
              in_quotes = False
              current = []

              for char in line:
                  if char == ',' and not in_quotes:
                      parts.append(''.join(current).strip())
                      current = []
                  elif char == '"':
                      in_quotes = not in_quotes
                  else:
                      current.append(char)

              if current:
                  parts.append(''.join(current).strip())

              return parts

    def mapper(self, _, line):
        if line.startswith("title"):
            return

        fields = self.split_properly(line)

        title = fields[0]
        genres = fields[2]
        air_date = fields[-3]
        air_time =int(fields[-2])


        if 90000 > air_time >= 70000 and \
           self.contains_genre(genres) and \
           self.contains_letter(title):

          yield (title, genres), air_date




    def reducer(self, key, air_date):
        dates_set = set()
        title=key[0]
        genres=key[1]
        genres_count = len(genres.split(','))


        for date in air_date:
            dates_set.add(date)

        count_dates = len(dates_set)
        yield (title, genres), (count_dates, genres_count)

    def steps(self):
        return [
            MRStep(mapper=self.mapper, reducer=self.reducer)
        ]

if __name__ == '__main__':
    CompShow.run()


In [ ]:
!python hw1_mrA_000000000_000000000.py 420k_daily_prog_data.csv -q

In [ ]:
%%file hw1_mrB_000000000_000000000.py

from mrjob.job import MRJob
from mrjob.step import MRStep

class BestShow(MRJob):

    def contains_letter(self, title):
        letters = ['j', 'q', 'z']
        title_lower = title.lower()
        for letter in letters:
            if letter in title_lower:
                return True
        return False

    def contains_genre(self, genre):
        genres_list = ['Sitcom', 'Talk', 'Politics', 'Spanish', 'Community', 'Martial arts']
        for g in genres_list:
            if g in genre:
                return True
        return False


    def split_properly(self, line):
              parts = []
              in_quotes = False
              current = []

              for char in line:
                  if char == ',' and not in_quotes:
                      parts.append(''.join(current).strip())
                      current = []
                  elif char == '"':
                      in_quotes = not in_quotes
                  else:
                      current.append(char)

              if current:
                  parts.append(''.join(current).strip())

              return parts

    def mapper(self, _, line):
        if line.startswith("title"):
            return

        fields = self.split_properly(line)

        title = fields[0]
        genres = fields[2]
        air_date = fields[-3]
        air_time =int(fields[-2])


        if 90000 > air_time >= 70000 and \
           self.contains_genre(genres) and \
           self.contains_letter(title):

          yield (title, genres), air_date



    def reducer(self, key, air_date):
        dates_set = set()
        title=key[0]
        genres=key[1]
        genres_count = len(genres.split(','))

        for date in air_date:
            dates_set.add(date)

        count_dates = len(dates_set)
        yield None, (title, genres, count_dates + genres_count)

    def reducer_find_max(self, _, pairs):
        max_pair = max(pairs, key=lambda pair: pair[2])
        yield (max_pair[0], max_pair[1]), max_pair[2]



    def steps(self):
        return [
            MRStep(mapper=self.mapper, reducer=self.reducer),
            MRStep(reducer=self.reducer_find_max)

        ]

if __name__ == '__main__':
    BestShow.run()


In [ ]:
!python hw1_mrB_000000000_000000000.py 420k_daily_prog_data.csv -q

### **Part 2 - Spark:**

In [ ]:
!pip install --force-reinstall pyspark==3.4
!pip install findspark

In [ ]:
from datetime import datetime
import pyspark
from pyspark.sql import SparkSession
from pyspark.mllib.random import RandomRDDs
from pyspark.sql.types import*
import pandas as pd

spark = SparkSession.builder.appName('00 the basics').getOrCreate()
sc = spark.sparkContext
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
file = sc.textFile("420k_daily_prog_data.csv")
header = file.first()
filtered_file_1 = file.filter(lambda line: line != header)
forbidden_words = ['big','the', 'bang', 'theory', 'almanac', 'met',
                   'mother', 'your', 'city', 'anatomy', 'game', 'thrones',
                   'guy', 'family', 'friends', 'senate', 'two']

def contain_words(line):
    title_words = line[0].split(" ")
    for s1 in title_words:
        for s2 in forbidden_words:
            if s1.lower() == s2:
                return False
    return True

def calculate_score(line):
    score = 0
    if len(line[2]) == 1:
        if line[2][0] == 'Sitcom': score += 7
        elif line[2][0] == 'Action': score += 90
        elif line[2][0] == 'Documentary': score += 90
    score += (float(line[5])/15)
    date_str = line[3]
    date_obj = datetime.strptime(date_str, '%Y%m%d')
    if date_obj.weekday() in [1, 3, 5, 6]:
        score += 3
    new_arr = (line[0], line[1], line[2], line[3], line[4], line[5], score)
    return new_arr

def split_properley(line):
    line = line.strip()
    title = None
    prog_code = None
    genres = None
    date = None
    air_time = None
    duration = None

    fields = []
    i = 0
    while i < len(line):
        if line[i] == '"':
            start = i + 1
            i = line.find('"', start)
            if i == -1:
                raise ValueError("Unmatched quotes in line")
            fields.append(line[start:i].replace('""', '"'))
            i += 1
        else:
            start = i
            i = line.find(',', start)
            if i == -1:
                i = len(line)
            fields.append(line[start:i].strip())
        i += 1

    if len(fields) >= 6:
        title = fields[0]
        prog_code = fields[1]
        genres = tuple(fields[2].split(',')) if fields[2] else ()
        date = fields[3]
        air_time = int(float(fields[4])) if fields[4] else None
        duration = int(float(fields[5])) if fields[5] else None

    return (title, prog_code, genres, date, air_time, duration)

filtered_file_2 = filtered_file_1.map(split_properley)
relevant_rows = filtered_file_2.filter(contain_words)
score_calculating = relevant_rows.map(calculate_score)
key_value_order = score_calculating.map(lambda x: ((x[0], x[2]), x[6]))
total_score_per_key = key_value_order.reduceByKey(lambda x, y: x + y)
results = total_score_per_key.collect()
top_20_tuples = sorted(results, key=lambda x: -x[1])[:20]


df = pd.DataFrame(top_20_tuples, columns=['title_genre', 'score'])
df['title_genre'] = df['title_genre'].apply(lambda x: f"{{{x[0]}, {', '.join(x[1])}}}")

print("+-------------------------------------------------+--------------------------------------------")
print("|title_genre                                                   | score ")
print("+-------------------------------------------------+--------------------------------------------")


for index, row in df.iterrows():
    print(f"| {row['title_genre']:<60} | {row['score']:<25} |")
